In [2]:
import sklearn as sl
import statsmodels as sm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import tensorly as tl


warnings.filterwarnings("ignore")

In [11]:

"""
HDGM univariato stimato con EM (Kalman filter + RTS smoother).

y(s,t) = X_beta(s,t)' beta + xi(s,t) + eps(s,t)
xi(s,t) = g * xi(s,t-1) + eta(s,t),  eta ~ GP(0, v*exp(-d/theta))
eps(s,t) ~ N(0, sigma2_eps)
n num stazioni, p num covariate, #T num periodi
"""

import numpy as np
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from scipy.special import erf
from scipy.stats import norm
import tensorly as tl

class HDGM:
    def __init__(self, max_iter=200, tol=1e-6, verbose=True, previsione = None, misure=None, metodo = 'gradiente'):
        self.max_iter, self.tol, self.verbose, self.previsione, self.misure, self.metodo = max_iter, tol, verbose, previsione, misure, metodo

    @staticmethod
    def _exp_corr(dist, theta):
        return np.exp(-dist / theta)

    def _sigma_eta(self, v, theta):
        return v * self._exp_corr(self.dist_, theta)

    def _kalman_filter(self, Y, X, beta, g, Sigma_eta, sigma2_eps, mu0, Sigma0):
        n, T = Y.shape
        a_pred = np.zeros((T, n)); P_pred = np.zeros((T, n, n))
        a_filt = np.zeros((T, n)); P_filt = np.zeros((T, n, n))
        loglik = 0.0
        a_prev, P_prev = mu0.copy(), Sigma0.copy()

        for t in range(T):
            a_p = a_prev if t == 0 else g * a_filt[t - 1]
            P_p = P_prev if t == 0 else g**2 * P_filt[t - 1] + Sigma_eta
            a_pred[t], P_pred[t] = a_p, P_p

            obs_idx = np.where(~np.isnan(Y[:, t]))[0]
            if len(obs_idx) == 0:
                a_filt[t], P_filt[t] = a_p, P_p
                continue

            resid = Y[obs_idx, t] - X[obs_idx, :, t] @ beta
            H = np.eye(n)[obs_idx]
            S = H @ P_p @ H.T + sigma2_eps * np.eye(len(obs_idx))
            S_inv = np.linalg.inv(S+1e-7*np.eye(S.shape[0]))
            K = P_p @ H.T @ S_inv

            innov = resid - H @ a_p
            a_filt[t] = a_p + K @ innov
            P_filt[t] = P_p - K @ H @ P_p

            sign, logdet = np.linalg.slogdet(S)
            loglik += -0.5 * (logdet + innov @ S_inv @ innov + len(obs_idx) * np.log(2 * np.pi))
        return a_pred, P_pred, a_filt, P_filt, loglik

    def _rts_smoother(self, a_pred, P_pred, a_filt, P_filt, g):
        T, n = a_filt.shape
        a_s = np.zeros_like(a_filt); P_s = np.zeros_like(P_filt)
        P_lag = np.zeros((T, n, n))
        a_s[-1], P_s[-1] = a_filt[-1], P_filt[-1]

        for t in range(T - 2, -1, -1):
            J = P_filt[t] @ (g * np.eye(n)).T @ np.linalg.inv(P_pred[t + 1]+1e-7*np.eye(P_pred[t+1].shape[0]))
            a_s[t] = a_filt[t] + J @ (a_s[t + 1] - a_pred[t + 1])
            P_s[t] = P_filt[t] + J @ (P_s[t + 1] - P_pred[t + 1]) @ J.T
            P_lag[t + 1] = J @ P_s[t + 1]
        return a_s, P_s, P_lag

    
    #modifica
    @staticmethod
    def check_station_distances(coords, nomi, threshold_km=1.0):
        dist = cdist(coords, coords)
        n = dist.shape[0]
        pairs = []
        for i in range(n):
            for j in range(i + 1, n):
                if dist[i, j] < threshold_km:
                    pairs.append((i, j, dist[i, j]))
        pairs.sort(key=lambda x: x[2])
 
        if pairs:
            print(f"[HDGM.check_station_distances] {len(pairs)} coppie sotto "
                  f"{threshold_km} km:")
            for i, j, d in pairs:
                print(f" stazioni {nomi[i]} - {nomi[j]}: {d:.3f} km")
        else:
            print(f"[HDGM.check_station_distances] nessuna coppia sotto "f"{threshold_km} km.")
        #return pairs



    

    def fit(self, X, Y, coords, g0=0.5, v0=1.0, theta0=None, sigma2_0=1.0, beta0=None):
        """
        X : (n, p, T)  -> X_beta(s,t)
        Y : (n, T)     -> osservazioni, np.nan dove mancanti
        coords : (n, 2)
        id_staz in [0,n], identifica la stazione da stimare tramite losocv
        """
        n, p, T = X.shape
        self.dist_ = cdist(coords, coords)
        if theta0 is None:
            theta0 = np.median(self.dist_[self.dist_ > 0]) / 2

        beta = beta0 if beta0 is not None else np.zeros(p)
        g, v, theta, sigma2_eps = g0, v0, theta0, sigma2_0
        loglik_old = -np.inf

        for it in range(self.max_iter):
            Sigma_eta = self._sigma_eta(v, theta)
            mu0 = np.zeros(n)
            Sigma0 = v / max(1 - g**2, 1e-6) * self._exp_corr(self.dist_, theta)

            a_pred, P_pred, a_filt, P_filt, loglik = self._kalman_filter(Y, X, beta, g, Sigma_eta, sigma2_eps, mu0, Sigma0)
            a_s, P_s, P_lag = self._rts_smoother(a_pred, P_pred, a_filt, P_filt, g)

            S11 = np.zeros((n, n)); S00 = np.zeros((n, n)); S10 = np.zeros((n, n))
            for t in range(1, T):
                S11 += P_s[t] + np.outer(a_s[t], a_s[t])
                S00 += P_s[t-1] + np.outer(a_s[t-1], a_s[t-1])
                S10 += P_lag[t] + np.outer(a_s[t], a_s[t-1])

            # beta via minimi quadrati sui residui (regressione su y - stato smussato)
            XtX = np.zeros((p, p)); Xty = np.zeros(p)
            for t in range(T):
                obs_idx = np.where(~np.isnan(Y[:, t]))[0]
                if len(obs_idx) == 0:
                    continue
                Xt = X[obs_idx, :, t]
                yt = Y[obs_idx, t] - a_s[t, obs_idx]
                XtX += Xt.T @ Xt
                Xty += Xt.T @ yt
                #if t%20==0:
                #    print(np.linalg.cond(XtX))
            beta = np.linalg.solve(XtX + 1e-5*np.eye(p), Xty) #per evitare singolarità

            Sigma_eta_inv = np.linalg.inv(Sigma_eta + 1e-7*np.eye(Sigma_eta.shape[0]))
            g = np.clip(np.trace(Sigma_eta_inv @ S10.T) / np.trace(Sigma_eta_inv @ S00),-0.995, 0.995)

            if 'simp' in self.metodo or 'Nelder-Mead' in self.metodo:
                def neg_Q2(par):
                    vv, th = np.exp(par[0]), np.exp(par[1])
                    Se = vv * self._exp_corr(self.dist_, th)
                    _, logdet = np.linalg.slogdet(Se)
                    Se_inv = np.linalg.inv(Se+1e-7*np.eye(Se.shape[0]))
                    Gm = g * np.eye(n)
                    M = S11 - S10 @ Gm.T - Gm @ S10.T + Gm @ S00 @ Gm.T
                    return (T - 1) * logdet + np.trace(Se_inv @ M)
                
                res = minimize(neg_Q2, x0=np.log([v, theta]), method="Nelder-Mead")

            elif 'grad' in self.metodo or 'Newton' in self.metodo:
                def neg_Q2(par, self, g, S11, S00, S10, n, T):
                    v, theta = np.exp(par[0]), np.exp(par[1])  #in log per vincoli > 0
                    R = self._exp_corr(self.dist_, theta)
                    Se = v * R 
                    Se_inv = np.linalg.inv(Se+1e-7*np.eye(Se.shape[0]))
                
                    Gm = g * np.eye(n)
                    M = S11 - S10 @ Gm.T - Gm @ S10.T + Gm @ S00 @ Gm.T
                
                    _, logdet = np.linalg.slogdet(Se)
                    Se_inv_M = Se_inv @ M
                    fval = (T - 1) * logdet + np.trace(Se_inv_M)
        
                    # derivate rispetto a v e theta (regola della catena per i parametri in scala log)
                    dSe_dv = R                                   # d Sigma_eta / d v
                    dR_dtheta = R * (self.dist_ / theta**2)      # d rho / d theta (correlazione esponenziale)
                    dSe_dtheta = v * dR_dtheta
                
                    # d/dphi [ (T-1)logdet(Se) + tr(Se^-1 M) ] = (T-1)tr(Se^-1 dSe) - tr(Se^-1 dSe Se^-1 M)
                    def dQ(dSe):
                        return (T - 1) * np.trace(Se_inv @ dSe) - np.trace(Se_inv @ dSe @ Se_inv_M)
                
                    grad_v = dQ(dSe_dv)
                    grad_theta = dQ(dSe_dtheta)
                
                    # regola della catena: par=log(v),log(theta)=> d/d(log v) = v*d/dv
                    grad = np.array([grad_v * v, grad_theta * theta])
                    return fval, grad
                
                res = minimize(neg_Q2, x0=np.log([v, theta]),args=(self, g, S11, S00, S10, n, T),method="Newton-CG", jac=True, tol= self.tol)

            v, theta = np.exp(res.x)

            num, den = 0.0, 0
            for t in range(T):
                obs_idx = np.where(~np.isnan(Y[:, t]))[0]
                if len(obs_idx) == 0:
                    continue
                resid = Y[obs_idx, t] - X[obs_idx, :, t] @ beta - a_s[t, obs_idx]
                num += np.sum(resid**2) + np.sum(np.diag(P_s[t])[obs_idx])
                den += len(obs_idx)
            sigma2_eps = num / den

            if self.verbose:
                print(f"iter {it:3d}  loglik={loglik:.3f}  g={g:.3f}  "
                      f"v={v:.3f}  theta={theta:.3f}  sigma2_eps={sigma2_eps:.3f}")
            if abs(loglik - loglik_old) < self.tol:
                break
            loglik_old = loglik

        self.a_pred = a_pred
        self.a_filt, self.P_filt = a_filt, P_filt
        self.beta_, self.g_, self.v_ = beta, g, v
        self.theta_, self.sigma2_eps_ = theta, sigma2_eps
        self.a_smooth_, self.P_smooth_, self.loglik_ = a_s, P_s, loglik
        self.coords_ = coords

    def _spatial_predict_latent_state(self, coords_new, jitter=1e-8, return_var=False):
        Sigma_ff = self.v_ * self._exp_corr(self.dist_, self.theta_)
        dist_0f = cdist(coords_new, self.coords_)
        Sigma_0f = self.v_ * self._exp_corr(dist_0f, self.theta_)

        jitter_abs = jitter * max(self.v_, 1e-12)
        Sigma_ff_reg = Sigma_ff + jitter_abs * np.eye(Sigma_ff.shape[0])
        weights = np.linalg.solve(Sigma_ff_reg, Sigma_0f.T).T  # (n_new, n_train)

        xi_hat = self.a_smooth_ @ weights.T  # (T, n_new)

        if not return_var:
            return xi_hat

        # varianza pura: incertezza spaziale anche se a_smooth_ fosse esatto
        var_pura = self.v_ - np.sum(weights * Sigma_0f, axis=1)          # (n_new,)
        var_pura = np.maximum(var_pura, 0)  # per sicurezza numerica

        # propagazione dell'incertezza del filtro (P_smooth_) attraverso i pesi di quadratura
        var_prop = np.einsum('ij,tjk,ik->ti', weights, self.P_smooth_, weights)  # (T, n_new)

        var_tot = var_pura[None, :] + var_prop   # (T, n_new)
        return xi_hat, var_tot



    def predict(self, X_new, coords_new, return_var=False):
        n_n, p, T = X_new.shape
        if return_var:
            xi_hat, var_xi = self._spatial_predict_latent_state(coords_new, return_var=True)
        else:
            xi_hat = self._spatial_predict_latent_state(coords_new)

        pred = np.zeros([n_n, T])
        for t in range(T):
            pred[:, t] = X_new[:, :, t] @ self.beta_ + xi_hat[t, :]

        if return_var:
            # var_xi (T, n_new), trasporre per avere le stesse dimensioni di pred (n_new, T)
            sigma2 = var_xi.T
            return pred, sigma2
        return pred

    
    @staticmethod
    def _r2(y_true, y_pred):
        valid = ~ np.isnan(y_true)
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        ss_res = np.sum((y_true[valid] - y_pred[valid]) ** 2)
        ss_tot = np.sum((y_true[valid] - np.mean(y_true[valid])) ** 2)
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    @classmethod
    def losocv(cls, X, Y, coords, station_ids=None, exclude_stations=None,init_kwargs=None, fit_kwargs=None, verbose=True):
        """
        Parametri
        X : (n, p, T)
        Y : (n, T)      np.nan dove mancante
        coords : (n, 2)
        station_ids : lista di n etichette 
        exclude_stations : lista di id/indici di stazioni da NON usare come per validazione (stazioni anomale), pur restando nell'addestramento
        init_kwargs : dict di argomenti per il costruttore HDGM(...) (max_iter, tol, metodo)
        fit_kwargs : dict di argomenti per model.fit() (g0, v0, theta0, sigma2_0, beta0)
        verbose : stampa avanzamento per stazione
 
        Restituisce dict con:
          'y_true', 'y_pred', 'station_of_obs' : vettori concatenati su tutte le previsioni fuori campione
          'metrics_global' : dict con MSE, RMSE, MAE, R2 aggregati
          'metrics_per_station' : dict {station_id: {MSE, RMSE, MAE, R2, n_obs}}
          'models' : dict {station_id: modello HDGM adattato escludendola}
        """
        n, p, T = X.shape
        init_kwargs = dict(init_kwargs or {})
        init_kwargs["verbose"] = False
        fit_kwargs = fit_kwargs or {}
        if station_ids is None:
            station_ids = list(range(n))
        exclude_stations = set(exclude_stations or [])
 
        all_y_true, all_y_pred, all_station = [], [], []
        per_station_metrics = {}
        fitted_models = {}
 
        for i, sid in enumerate(station_ids):
            if sid in exclude_stations:
                if verbose:
                    print(f"[LOSOCV] stazione {sid}: esclusa dalla validazione (saltare)")
                continue
 
            train_idx = [j for j in range(n) if j != i]
            X_train, Y_train, coords_train = X[train_idx], Y[train_idx], coords[train_idx]
            X_test, coords_test = X[[i]], coords[[i]]
            y_test = Y[i, :]
 
            if verbose:
                print(f"[LOSOCV] stazione {sid} ({i+1}/{n}): stima su {len(train_idx)} stazioni...")
 
            model = cls(**init_kwargs)
            model.fit(X_train, Y_train, coords_train, **fit_kwargs)
            fitted_models[sid] = model
 
            y_pred = model.predict(X_test, coords_test)[0, :]
 
            if not np.all(np.isfinite(y_pred)):
                print(f"[LOSOCV] stazione {sid}: previsione non finita " f"(v_={model.v_:.4g}, theta_={model.theta_:.4g}, g_={model.g_:.4g}) " f"-> stazione saltata per evitare di corrompere l'R2 aggregato")
                continue
 
            obs_idx = np.where(~np.isnan(y_test))[0]
            if len(obs_idx) == 0:
                if verbose:
                    print(f"[LOSOCV] stazione {sid}: nessuna osservazione valida, saltare")
                continue
 
            y_t, y_p = y_test[obs_idx], y_pred[obs_idx]
            all_y_true.append(y_t)
            all_y_pred.append(y_p)
            all_station.extend([sid] * len(obs_idx))
 
            resid = y_t - y_p
            mse_s = np.mean(resid ** 2)
            per_station_metrics[sid] = {
                "MSE": mse_s,
                "RMSE": np.sqrt(mse_s),
                "MAE": np.mean(np.abs(resid)),
                "R2": cls._r2(y_t, y_p),
                "n_obs": len(obs_idx),}
            if verbose:
                print(f"    -> RMSE={per_station_metrics[sid]['RMSE']:.3f}  "
                      f"R2={per_station_metrics[sid]['R2']:.3f}")
 
        y_true_all = np.concatenate(all_y_true)
        y_pred_all = np.concatenate(all_y_pred)
        resid_all = y_true_all - y_pred_all
 
        mse = np.mean(resid_all ** 2)
        metrics_global = {
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "MAE": np.mean(np.abs(resid_all)),
            "R2": cls._r2(y_true_all, y_pred_all),
            "n_obs": len(y_true_all),
            "n_stations": len(per_station_metrics),}
 
        if verbose:
            print("\n[LOSOCV] risultati aggregati:")
            for k, v in metrics_global.items():
                print(f"  {k}: {v}")
 
        return {
            "y_true": y_true_all,
            "y_pred": y_pred_all,
            "station_of_obs": np.array(all_station),
            "metrics_global": metrics_global,
            "metrics_per_station": per_station_metrics,
            "models": fitted_models,}

    
    def _full_loglik(self, params, X, Y):
        """Log-verosimiglianza marginale (filtro Kalman) in funzione
        del vettore completo di parametri trasformati [beta..., log(g_trans), log(v), log(theta), log(sigma2_eps)],
        g viene ri-parametrizzato con atanh per mapparlo su tutto (-inf,+inf)."""
        p = X.shape[1]
        beta = params[:p]
        g = np.tanh(params[p]) # g in (-1, 1)
        v, theta, sigma2_eps = np.exp(params[p+1]), np.exp(params[p+2]), np.exp(params[p+3])

        Sigma_eta = self._sigma_eta(v, theta)
        mu0 = np.zeros(X.shape[0])
        Sigma0 = v / max(1 - g**2, 1e-6) * self._exp_corr(self.dist_, theta)
        _, _, _, _, loglik = self._kalman_filter(Y, X, beta, g, Sigma_eta, sigma2_eps, mu0, Sigma0)
        return loglik

    def summary(self, X, Y, param_names=None, rel_step=1e-5):
        """
        Calcola SE, z-stat e p-value (Wald) per tutti i parametri
        [beta..., log(g_trans), log(v), log(theta), log(sigma2_eps)]
        usando l'inversa della matrice di informazione osservata,
        ottenuta come Hessiana numerica della log-verosimiglianza.
        """

        p = X.shape[1]
        u0 = np.concatenate([self.beta_,[np.arctanh(np.clip(self.g_, -0.999, 0.999)),np.log(self.v_), np.log(self.theta_), np.log(self.sigma2_eps_)] ])
        k = len(u0)
        steps = rel_step * np.maximum(np.abs(u0), 1e-2)

        f0 = self._full_loglik(u0, X, Y)
        H = np.zeros((k, k))
        for i in range(k):
            ei = np.zeros(k); ei[i] = steps[i]
            fpp = self._full_loglik(u0 + ei, X, Y)
            fmm = self._full_loglik(u0 - ei, X, Y)
            H[i, i] = (fpp - 2*f0 + fmm) / steps[i]**2
            for j in range(i+1, k):
                ej = np.zeros(k); ej[j] = steps[j]
                fpp2 = self._full_loglik(u0+ei+ej, X, Y)
                fpm2 = self._full_loglik(u0+ei-ej, X, Y)
                fmp2 = self._full_loglik(u0-ei+ej, X, Y)
                fmm2 = self._full_loglik(u0-ei-ej, X, Y)
                H[i, j] = H[j, i] = (fpp2-fpm2-fmp2+fmm2) / (4*steps[i]*steps[j])

        cov_u = np.linalg.inv(-H+1e-8*np.eye(H.shape[0]))          # covarianza sulla scala non vincolata
        se_u = np.sqrt(np.maximum(np.diag(cov_u), 0))

        # metodo delta: Var(phi) ~= (d phi/d u)^2 * Var(u)
        # derivate delle trasformazioni inverse valutate in u0
        jac = np.ones(k)
        jac[p] = 1 - np.tanh(u0[p])**2          # d/du[tanh(u)] per g
        jac[p+1] = np.exp(u0[p+1])              # d/du[exp(u)] per v
        jac[p+2] = np.exp(u0[p+2])              # per theta
        jac[p+3] = np.exp(u0[p+3])              # per sigma2_eps

        se_original = np.abs(jac) * se_u
        params_original = np.concatenate([self.beta_, [self.g_, self.v_, self.theta_, self.sigma2_eps_]])

        z = params_original / se_original
        from scipy.stats import norm
        pvals = 2 * (1 - norm.cdf(np.abs(z)))

        if param_names is None:
            param_names = [f"beta_{i}" for i in range(p)] + ["g", "v", "theta", "sigma2_eps"]

        print(f"{'Parametro':<12}{'Stima':>10}{'SE':>10}{'z':>10}{'p-value':>12}")
        
        for name, est, s, zz, pv in zip(param_names, params_original, se_original, z, pvals):
            stars = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else ""
            print(f"{name:<12}{est:>10.4f}{s:>10.4f}{zz:>10.3f}{pv:>12.4g} {stars}")

        #return se_original, z, pvals, cov_u

In [4]:
#definizione dei dati
percorso = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/completi.xlsx'
c_staz = 'stazione'
c_data = 'data'
c_geo = ['comune','provincia','latitudine','longitudine']
c_misure = ['u100_media','v100_media','tp_media','t2m_media','blh_media','bovini',	'suini',	'veg_alta',	'veg_bassa'	,'rh']
c_previsione = 'PM10_media'
freq = 'D'

tab = pd.read_excel(percorso)
tab = tab.sort_values([c_staz,c_data])
#print(tab.describe().T)

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)

percorso_prov = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/id stazioni.xlsx'

tab_staz = pd.read_excel(percorso_prov)
tab_staz = tab_staz.sort_values("nome staz").reset_index(drop=True)
id_staz = list(tab_staz["nome staz"].values)


In [4]:
#costruzione della matrice dei coefficienti con le covariate suggerite dall'articolo
#c_misure = ['PM2o5_media',	'PM10_media','NO_media','NO2_media','NOX_media', 'CO_media','O3_media','SO2_media','BENZENE_media','u10_media','v10_media','u100_media','v100_media','ssrd_media','tp_media','d2m_media','t2m_media','blh_media','sp_media',	'bovini',	'ovini',	'suini',	'pollame',	'veg_alta',	'veg_bassa'	,'ettari_consumati',	'perc_suolo','rh']
#10 covariate + 1 intercetta per prevedere PM10
c_misure = ['u100_media','v100_media','tp_media','t2m_media','blh_media','bovini',	'suini',	'veg_alta',	'veg_bassa'	,'rh']
c_previsione = 'PM10_media'

stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni) #64
n_regr = len(c_misure) #10
periodo = len(istanti) #1826

c_misure.remove('u100_media')
c_misure.remove('v100_media')

X_b = tl.zeros([n_staz, n_regr,periodo])
y_vera = tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])
for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_vera[ind_s, :] = tab[tab['stazione']==staz][c_previsione].to_numpy()
        
    vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
        
    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-1,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    X_b[ind_s,-1,:] = vel100.to_numpy()

#trasformazione logaritmica per garantire positività
y_vera = np.log(y_vera)

In [6]:
def coords2km(coords_deg):
    R = 6371.0  # raggio terrestre medio in km
    lon = np.radians(coords_deg[:, 0])
    lat = np.radians(coords_deg[:, 1])
    lat0 = np.mean(lat)
 
    x_km = R * np.cos(lat0) * lon
    y_km = R * lat
    return np.column_stack([x_km, y_km])

percorso_prov = 'C:/Users/CdC/Desktop/studium/tirocinio/dati/puliti/id stazioni.xlsx'

tab_staz = pd.read_excel(percorso_prov)
tab_staz = tab_staz.sort_values("nome staz").reset_index(drop=True)

id_staz = list(tab_staz["nome staz"].values)

coords_km = coords2km(coord)
HDGM.check_station_distances(coords_km,id_staz, threshold_km=2.5)

[HDGM.check_station_distances] 21 coppie sotto 2.5 km:
 stazioni machiavelli - tamburi: 0.634 km
 stazioni fanin - minzoni: 0.768 km
 stazioni casardi - via_trani: 0.792 km
 stazioni archimede - tamburi: 0.809 km
 stazioni perrino - via_taranto: 0.834 km
 stazioni archimede - machiavelli: 1.015 km
 stazioni mille - via_taranto: 1.176 km
 stazioni monopoli - monopoli_italgreen: 1.293 km
 stazioni casale - mille: 1.300 km
 stazioni garigliano - libertini: 1.354 km
 stazioni casale - via_taranto: 1.787 km
 stazioni perrino - terminal: 1.872 km
 stazioni caldarola - cavour: 1.958 km
 stazioni casardi - pini: 1.987 km
 stazioni cappuccini - mille: 2.002 km
 stazioni mille - perrino: 2.008 km
 stazioni colacem - galatina: 2.041 km
 stazioni terminal - via_taranto: 2.092 km
 stazioni casale - terminal: 2.144 km
 stazioni perrino - sisri: 2.360 km
 stazioni casale - perrino: 2.408 km


In [26]:
#adattamento del modello
modello = HDGM(max_iter=500,tol = 1e-6, previsione= c_previsione, misure = ['1']+ c_misure+ ['vel100'],metodo = 'grad')
modello.fit(X_b, y_vera, coord)
modello.summary(X_b,y_vera, param_names = ['1']+ c_misure+ ['vel100',"g", "v", "theta", "sigma2_eps"])

iter   0  loglik=-133317.925  g=0.667  v=1.226  theta=0.623  sigma2_eps=0.299
iter   1  loglik=-87263.133  g=0.696  v=0.949  theta=0.693  sigma2_eps=0.142
iter   2  loglik=-61933.256  g=0.741  v=0.685  theta=0.738  sigma2_eps=0.082
iter   3  loglik=-43170.168  g=0.797  v=0.461  theta=0.712  sigma2_eps=0.054
iter   4  loglik=-29555.980  g=0.853  v=0.300  theta=0.626  sigma2_eps=0.039
iter   5  loglik=-20362.723  g=0.899  v=0.206  theta=0.547  sigma2_eps=0.032
iter   6  loglik=-14766.288  g=0.928  v=0.160  theta=0.518  sigma2_eps=0.028
iter   7  loglik=-11761.562  g=0.943  v=0.139  theta=0.533  sigma2_eps=0.027
iter   8  loglik=-10174.892  g=0.949  v=0.130  theta=0.575  sigma2_eps=0.026
iter   9  loglik=-9174.570  g=0.951  v=0.125  theta=0.628  sigma2_eps=0.027
iter  10  loglik=-8413.418  g=0.953  v=0.122  theta=0.686  sigma2_eps=0.027
iter  11  loglik=-7779.472  g=0.954  v=0.119  theta=0.745  sigma2_eps=0.028
iter  12  loglik=-7234.373  g=0.955  v=0.117  theta=0.804  sigma2_eps=0.028
it

In [28]:
y_previsione = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione[:,t] = X_b[:,:,t]@modello.beta_+modello.a_filt[t]

#XtX mal condizionata
valid = ~np.isnan(y_vera)
R2 = 1- np.sum((y_vera[valid]-y_previsione[valid])**2)/np.sum((y_vera[valid]-np.mean(y_vera[valid]))**2)
print(R2)

0.8720626513055358


In [24]:
#costruzione della matrice dei coefficienti con tutte le covariate

c_misure = ['u10_media','v10_media','u100_media','v100_media','ssrd_media','tp_media','d2m_media','t2m_media','blh_media','sp_media',	'bovini',	'ovini',	'suini',	'pollame',	'veg_alta',	'veg_bassa'	,'ettari_consumati','perc_suolo','rh']

stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)
c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')

n_regr = len(c_misure)

X_b = tl.zeros([n_staz, n_regr+1+2,periodo])
y_vera = tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])


for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_vera[ind_s, :] = tab[tab['stazione']==staz][c_previsione].to_numpy()

    vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-2,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    X_b[ind_s,-2,:] = vel10.to_numpy()
    X_b[ind_s,-1,:] = vel100.to_numpy()

#trasformazione logaritmica per garantire positività
y_vera = np.log(y_vera)


In [25]:
#adattamento del modello
modello = HDGM(max_iter=500,tol=1e-6, previsione= c_previsione, misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo = 'grad')
modello.fit(X_b, y_vera, coord)

modello.summary(X_b,y_vera, param_names = ['1']+ c_misure+ ['vel10','vel100']+["g", "v", "theta", "sigma2_eps"])

iter   0  loglik=-133317.925  g=0.667  v=1.226  theta=0.623  sigma2_eps=0.298
iter   1  loglik=-87140.188  g=0.695  v=0.943  theta=0.689  sigma2_eps=0.142
iter   2  loglik=-61845.008  g=0.741  v=0.679  theta=0.732  sigma2_eps=0.082
iter   3  loglik=-43106.391  g=0.797  v=0.457  theta=0.704  sigma2_eps=0.054
iter   4  loglik=-29505.107  g=0.853  v=0.297  theta=0.620  sigma2_eps=0.039
iter   5  loglik=-20320.244  g=0.899  v=0.204  theta=0.543  sigma2_eps=0.032
iter   6  loglik=-14736.033  g=0.928  v=0.159  theta=0.515  sigma2_eps=0.028
iter   7  loglik=-11741.822  g=0.942  v=0.138  theta=0.531  sigma2_eps=0.027
iter   8  loglik=-10158.829  g=0.948  v=0.129  theta=0.573  sigma2_eps=0.026
iter   9  loglik=-9158.997  g=0.951  v=0.124  theta=0.626  sigma2_eps=0.027
iter  10  loglik=-8397.800  g=0.953  v=0.121  theta=0.683  sigma2_eps=0.027
iter  11  loglik=-7763.827  g=0.954  v=0.119  theta=0.742  sigma2_eps=0.028
iter  12  loglik=-7218.729  g=0.955  v=0.117  theta=0.800  sigma2_eps=0.028
it

In [26]:
y_previsione = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione[:,t] = X_b[:,:,t]@modello.beta_+modello.a_filt[t,:n_staz]

valid = ~np.isnan(y_vera)
R2 = 1- np.sum((y_vera[valid]-y_previsione[valid])**2)/np.sum((y_vera[valid]-np.mean(y_vera[valid]))**2)
print(R2)

0.8719659238761694


In [18]:
#costruzione della matrice dei coefficienti con le covariate significative (p<0.05), eliminando le correlate, e inserendo regressori categorici per le date

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab_mens = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)


c_misure = ['u10_media','v10_media','tp_media','d2m_media','rh']

stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)
c_misure.remove('u10_media')
c_misure.remove('v10_media')

n_regr = len(c_misure)

X_b = tl.zeros([n_staz, n_regr+1+1+11,periodo])
y_vera = tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])

for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_vera[ind_s, :] = tab[tab['stazione']==staz][c_previsione].to_numpy()

    vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-12,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    X_b[ind_s,-12,:] = vel10.to_numpy()
    #X_b[ind_s,-12,:] = vel100.to_numpy()
    X_b[ind_s,-11:,:] = regr_mens.to_numpy().transpose()[:,:1826]

#trasformazione logaritmica per garantire positività
y_vera = np.log(y_vera)


In [19]:
#adattamento del modello su PM10
modello = HDGM(max_iter=1000, tol = 1e-6, previsione= c_previsione, misure = ['1']+ c_misure+ ['vel10'] + col_mens,metodo='gradiente')
modello.fit(X_b, y_vera, coord)

modello.summary(X_b,y_vera, param_names = ['1']+ c_misure+ ['vel10']+col_mens+ ["g", "v", "theta", "sigma2_eps"])

iter   0  loglik=-133317.925  g=0.667  v=1.226  theta=0.623  sigma2_eps=0.300
iter   1  loglik=-87452.271  g=0.698  v=0.950  theta=0.693  sigma2_eps=0.143
iter   2  loglik=-62096.053  g=0.744  v=0.682  theta=0.733  sigma2_eps=0.082
iter   3  loglik=-43305.804  g=0.800  v=0.457  theta=0.702  sigma2_eps=0.054
iter   4  loglik=-29643.301  g=0.857  v=0.295  theta=0.613  sigma2_eps=0.039
iter   5  loglik=-20398.246  g=0.902  v=0.202  theta=0.536  sigma2_eps=0.032
iter   6  loglik=-14782.030  g=0.930  v=0.158  theta=0.510  sigma2_eps=0.028
iter   7  loglik=-11785.385  g=0.944  v=0.138  theta=0.529  sigma2_eps=0.027
iter   8  loglik=-10203.765  g=0.950  v=0.129  theta=0.572  sigma2_eps=0.026
iter   9  loglik=-9201.557  g=0.952  v=0.125  theta=0.626  sigma2_eps=0.027
iter  10  loglik=-8436.970  g=0.954  v=0.121  theta=0.684  sigma2_eps=0.027
iter  11  loglik=-7799.756  g=0.955  v=0.119  theta=0.743  sigma2_eps=0.028
iter  12  loglik=-7251.626  g=0.956  v=0.117  theta=0.802  sigma2_eps=0.028
it

In [20]:
y_previsione = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione[:,t] = X_b[:,:,t]@modello.beta_ +modello.a_filt[t]

valid = ~np.isnan(y_vera)
R2 = 1- np.sum((y_vera[valid]-y_previsione[valid])**2)/np.sum((y_vera[valid]-np.mean(y_vera[valid]))**2)
RMSE = np.sqrt(np.sum((y_vera[valid]-y_previsione[valid])**2)/len(y_vera[valid]))
MAE = np.sum(np.abs(y_vera[valid]-y_previsione[valid]))/len(y_vera[valid])
print("R2=",R2)
print("RMSE=",RMSE)
print("MAE=",MAE)

R2= 0.8720143960592117
RMSE= 0.17651911863937764
MAE= 0.12127479633948508


In [ ]:
#LOSOCV per la valutazione del modello PM10
res = HDGM.losocv(X_b,y_vera,coord,station_ids= id_staz,init_kwargs=dict(max_iter=200, tol=1e-6, metodo='gradiente'),fit_kwargs=dict(g0=0.5, v0=1.0, sigma2_0=1.0),)
print(res["metrics_global"])

[LOSOCV] stazione adige (1/64): stima su 63 stazioni...
    -> RMSE=0.198  R2=0.771
[LOSOCV] stazione altamura (2/64): stima su 63 stazioni...
    -> RMSE=0.211  R2=0.754
[LOSOCV] stazione andria (3/64): stima su 63 stazioni...
    -> RMSE=0.322  R2=0.580
[LOSOCV] stazione archimede (4/64): stima su 63 stazioni...
    -> RMSE=0.249  R2=0.589
[LOSOCV] stazione arnesano (5/64): stima su 63 stazioni...
    -> RMSE=0.315  R2=0.637
[LOSOCV] stazione azienda_russo (6/64): stima su 63 stazioni...
    -> RMSE=0.328  R2=0.617
[LOSOCV] stazione baldassarre (7/64): stima su 63 stazioni...
    -> RMSE=0.383  R2=0.476
[LOSOCV] stazione bitonto (8/64): stima su 63 stazioni...
    -> RMSE=0.318  R2=0.652
[LOSOCV] stazione caldarola (9/64): stima su 63 stazioni...
    -> RMSE=0.174  R2=0.821
[LOSOCV] stazione campi (10/64): stima su 63 stazioni...
    -> RMSE=0.250  R2=0.702
[LOSOCV] stazione candela_ex_comes (11/64): stima su 63 stazioni...
    -> RMSE=0.459  R2=0.450
[LOSOCV] stazione candela_scuola

In [28]:
#costruzione della matrice dei coefficienti con le covariate significative (p<0.05) e covariate plausibili per le emissioni di ammoniaca degli allevamenti, per inquinanti azotati

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','ovini','rh']
c_previsione = 'NOX_media'

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab_mens = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)

stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)
# c_misure.remove('u100_media')
# c_misure.remove('v100_media')
# c_misure.remove('u10_media')
# c_misure.remove('v10_media')

n_regr = len(c_misure)

X_b = tl.zeros([n_staz, n_regr+1+11,periodo])
y_vera = tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])


for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_vera[ind_s, :] = tab[tab['stazione']==staz][c_previsione].to_numpy()

    #vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    #vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-11,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    #X_b[ind_s,-13,:] = vel10.to_numpy()
    #X_b[ind_s,-12,:] = vel100.to_numpy()
    X_b[ind_s,-11:,:] = regr_mens.to_numpy().transpose()[:,:1826]

#trasformazione logaritmica per eliminare il vincolo di positività
y_vera = np.log(y_vera)

In [29]:
#adattamento del modello su NOX
modello = HDGM(max_iter=500, tol = 1e-6, previsione= c_previsione, misure = ['1']+ c_misure+col_mens,metodo='gradiente')
modello.fit(X_b, y_vera, coord)
modello.summary(X_b,y_vera, param_names = ['1']+ c_misure+col_mens+["g", "v", "theta", "sigma2_eps"])

iter   0  loglik=-141092.981  g=0.683  v=1.118  theta=0.547  sigma2_eps=0.378
iter   1  loglik=-103575.186  g=0.799  v=0.620  theta=0.371  sigma2_eps=0.190
iter   2  loglik=-79130.797  g=0.899  v=0.333  theta=0.249  sigma2_eps=0.110
iter   3  loglik=-59238.911  g=0.947  v=0.220  theta=0.214  sigma2_eps=0.073
iter   4  loglik=-46966.504  g=0.964  v=0.176  theta=0.221  sigma2_eps=0.055
iter   5  loglik=-40602.753  g=0.970  v=0.157  theta=0.246  sigma2_eps=0.046
iter   6  loglik=-37404.018  g=0.972  v=0.148  theta=0.277  sigma2_eps=0.042
iter   7  loglik=-35769.238  g=0.973  v=0.142  theta=0.306  sigma2_eps=0.040
iter   8  loglik=-34850.021  g=0.974  v=0.138  theta=0.332  sigma2_eps=0.040
iter   9  loglik=-34240.613  g=0.975  v=0.134  theta=0.355  sigma2_eps=0.040
iter  10  loglik=-33771.210  g=0.976  v=0.130  theta=0.376  sigma2_eps=0.041
iter  11  loglik=-33377.210  g=0.977  v=0.126  theta=0.395  sigma2_eps=0.042
iter  12  loglik=-33033.971  g=0.978  v=0.123  theta=0.413  sigma2_eps=0.0

In [30]:
y_previsione = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione[:,t] = X_b[:,:,t]@modello.beta_+modello.a_filt[t]

valid = ~np.isnan(y_vera)
R2 = 1- np.sum((y_vera[valid]-y_previsione[valid])**2)/np.sum((y_vera[valid]-np.mean(y_vera[valid]))**2)
RMSE = np.sqrt(np.sum((y_vera[valid]-y_previsione[valid])**2)/len(y_vera[valid]))
MAE = np.sum(np.abs(y_vera[valid]-y_previsione[valid]))/len(y_vera[valid])
print("R2=",R2)
print("RMSE=",RMSE)
print("MAE=",MAE)

R2= 0.9191840007767643
RMSE= 0.20540786776845604
MAE= 0.14999996738404


In [17]:
#confronti tra polveri sottili
c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','t2m_media','blh_media','bovini','veg_alta','veg_bassa','rh']

#costruzione della matrice dei coefficienti con le covariate significative (p<0.05) e inserendo regressori categorici per le date

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab_mens = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)


stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)

n_regr = len(c_misure)

X_b = tl.zeros([n_staz, n_regr+1+11,periodo])
y_2o5 = tl.zeros([n_staz, periodo])
y_10 =  tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])


for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_2o5[ind_s, :] = tab[tab['stazione']==staz]['PM2o5_media'].to_numpy()
    y_10[ind_s, :] = tab[tab['stazione']==staz]['PM10_media'].to_numpy()

    #vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    #vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-11,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    #X_b[ind_s,-13,:] = vel10.to_numpy()
    #X_b[ind_s,-12,:] = vel100.to_numpy()
    X_b[ind_s,-11:,:] = regr_mens.to_numpy().transpose()[:,:1826]

#trasformazione logaritmica per garantire positività
y_2o5 = np.log(y_2o5)
y_10 = np.log(y_10)

In [19]:
print('modello per PM2.5')
modello_2o5 = HDGM(max_iter=500, tol = 1e-6, previsione= 'PM2o5_media', misure = ['1']+ c_misure+col_mens,metodo='gradiente')
modello_2o5.fit(X_b, y_2o5, coord)
modello_2o5.summary(X_b,y_2o5, param_names = ['1']+ c_misure+col_mens+["g", "v", "theta", "sigma2_eps"])


print('modello per PM10')
modello_10 = HDGM(max_iter=500, tol = 1e-6, previsione= 'PM10_media', misure = ['1']+ c_misure+col_mens,metodo='gradiente')
modello_10.fit(X_b, y_10, coord)
modello_10.summary(X_b,y_10, param_names = ['1']+ c_misure+col_mens+["g", "v", "theta", "sigma2_eps"])


modello per PM2.5
iter   0  loglik=-75359.767  g=0.607  v=1.163  theta=0.551  sigma2_eps=0.342
iter   1  loglik=-54723.471  g=0.654  v=0.928  theta=0.519  sigma2_eps=0.174
iter   2  loglik=-44025.500  g=0.710  v=0.690  theta=0.464  sigma2_eps=0.106
iter   3  loglik=-36143.091  g=0.773  v=0.492  theta=0.396  sigma2_eps=0.072
iter   4  loglik=-29636.540  g=0.836  v=0.345  theta=0.328  sigma2_eps=0.052
iter   5  loglik=-24094.140  g=0.887  v=0.250  theta=0.277  sigma2_eps=0.040
iter   6  loglik=-19701.971  g=0.922  v=0.195  theta=0.248  sigma2_eps=0.033
iter   7  loglik=-16707.075  g=0.942  v=0.164  theta=0.239  sigma2_eps=0.029
iter   8  loglik=-14885.240  g=0.953  v=0.147  theta=0.243  sigma2_eps=0.026
iter   9  loglik=-13777.871  g=0.959  v=0.137  theta=0.255  sigma2_eps=0.025
iter  10  loglik=-13042.130  g=0.962  v=0.132  theta=0.272  sigma2_eps=0.024
iter  11  loglik=-12494.114  g=0.964  v=0.128  theta=0.291  sigma2_eps=0.023
iter  12  loglik=-12046.335  g=0.964  v=0.125  theta=0.313

In [22]:
y_prev_2o5 = tl.zeros([n_staz,periodo])
y_prev_10 = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_prev_2o5[:,t] = X_b[:,:,t]@modello_2o5.beta_+modello_2o5.a_filt[t]
    y_prev_10[:,t] = X_b[:,:,t]@modello_10.beta_+modello_10.a_filt[t]

valid_2o5 = ~np.isnan(y_2o5)
valid_10 = ~np.isnan(y_10)

R2_2o5 = 1- np.sum((y_2o5[valid_2o5]-y_prev_2o5[valid_2o5])**2)/np.sum((y_2o5[valid_2o5]-np.mean(y_2o5[valid_2o5]))**2)
RMSE_2o5 = np.sqrt(np.sum((y_2o5[valid_2o5]-y_prev_2o5[valid_2o5])**2)/len(y_2o5[valid_2o5]))
MAE_2o5 = np.sum(np.abs(y_2o5[valid_2o5]-y_prev_2o5[valid_2o5]))/len(y_2o5[valid_2o5])
print("Indici per PM2.5")
print("R2=",R2_2o5)
print("RMSE=",RMSE_2o5)
print("MAE=",MAE_2o5)

R2_10 = 1- np.sum((y_10[valid_10]-y_prev_10[valid_10])**2)/np.sum((y_10[valid_10]-np.mean(y_10[valid_10]))**2)
RMSE_10 = np.sqrt(np.sum((y_10[valid_10]-y_prev_10[valid_10])**2)/len(y_10[valid_10]))
MAE_10 = np.sum(np.abs(y_10[valid_10]-y_prev_10[valid_10]))/len(y_10[valid_10])
print("Indici per PM10")
print("R2=",R2_10)
print("RMSE=",RMSE_10)
print("MAE=",MAE_10)

Indici per PM2.5
R2= 0.8909365408096469
RMSE= 0.17714101498876222
MAE= 0.11997293878387608
Indici per PM10
R2= 0.8721934845077511
RMSE= 0.17639557505084735
MAE= 0.1212422646121263


In [33]:
#confronti tra inquinanti, PM2.5

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','veg_alta','veg_bassa','perc_suolo','rh']

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab_mens = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)


stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)
c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')

n_regr = len(c_misure)

X_b = tl.zeros([n_staz, n_regr+1+2+11,periodo])
y_2o5 = tl.zeros([n_staz, periodo])
y_10 = tl.zeros([n_staz, periodo])
y_co=  tl.zeros([n_staz, periodo])
y_nox =  tl.zeros([n_staz, periodo])
y_benzene = tl.zeros([n_staz, periodo])
y_o3=  tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])


for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_2o5[ind_s, :] = tab[tab['stazione']==staz]['PM2o5_media'].to_numpy()
    y_10[ind_s, :] = tab[tab['stazione']==staz]['PM10_media'].to_numpy()
    y_nox[ind_s, :] = tab[tab['stazione']==staz]['NOX_media'].to_numpy()
    y_benzene[ind_s, :] = tab[tab['stazione']==staz]['BENZENE_media'].to_numpy()
    y_co[ind_s, :] = tab[tab['stazione']==staz]['CO_media'].to_numpy()
    y_o3[ind_s, :] = tab[tab['stazione']==staz]['O3_media'].to_numpy()

    vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-13,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    X_b[ind_s,-13,:] = vel10.to_numpy()
    X_b[ind_s,-12,:] = vel100.to_numpy()
    X_b[ind_s,-11:,:] = regr_mens.to_numpy().transpose()[:,:1826]

y_2o5 = np.log(y_2o5)
y_10 =np.log(y_10)
y_co=  np.log(y_co)
y_nox =  np.log(y_nox)
y_benzene = np.log(y_benzene)
y_o3= np.log(y_o3)

In [34]:
print('PM10')
modello_10 = HDGM(max_iter=500, tol = 1e-6, previsione= 'PM10_media', misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo='gradiente')
modello_10.fit(X_b, y_10, coord)
modello_10.summary(X_b,y_10, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('NOX')
modello_nox = HDGM(max_iter=500, tol = 1e-6, previsione= 'NOX_media', misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo='gradiente')
modello_nox.fit(X_b, y_nox, coord)
modello_nox.summary(X_b,y_nox, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('CO')
modello_co = HDGM(max_iter=500, tol = 1e-6, previsione= 'CO_media', misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo='gradiente')
modello_co.fit(X_b, y_co, coord)
modello_co.summary(X_b,y_co, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('Benzene')
modello_benzene = HDGM(max_iter=500, tol = 1e-6, previsione= 'BENZENE_media', misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo='gradiente')
modello_benzene.fit(X_b, y_benzene, coord)
modello_benzene.summary(X_b,y_benzene, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('O3')
modello_o3 = HDGM(max_iter=500, tol = 1e-6, previsione= 'O3_media', misure = ['1']+ c_misure+ ['vel10', 'vel100'],metodo='gradiente')
modello_o3.fit(X_b, y_o3, coord)
modello_o3.summary(X_b,y_o3, param_names = ['1']+ c_misure+['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])



PM10
iter   0  loglik=-133317.925  g=0.667  v=1.226  theta=0.623  sigma2_eps=0.298
iter   1  loglik=-87234.116  g=0.696  v=0.946  theta=0.691  sigma2_eps=0.142
iter   2  loglik=-61922.940  g=0.741  v=0.681  theta=0.733  sigma2_eps=0.082
iter   3  loglik=-43170.107  g=0.798  v=0.457  theta=0.704  sigma2_eps=0.054
iter   4  loglik=-29550.994  g=0.854  v=0.297  theta=0.618  sigma2_eps=0.039
iter   5  loglik=-20344.567  g=0.900  v=0.203  theta=0.540  sigma2_eps=0.032
iter   6  loglik=-14744.049  g=0.929  v=0.158  theta=0.512  sigma2_eps=0.028
iter   7  loglik=-11744.424  g=0.943  v=0.138  theta=0.528  sigma2_eps=0.027
iter   8  loglik=-10160.475  g=0.949  v=0.129  theta=0.569  sigma2_eps=0.026
iter   9  loglik=-9159.527  g=0.952  v=0.124  theta=0.622  sigma2_eps=0.027
iter  10  loglik=-8396.807  g=0.953  v=0.121  theta=0.679  sigma2_eps=0.027
iter  11  loglik=-7761.147  g=0.954  v=0.118  theta=0.738  sigma2_eps=0.028
iter  12  loglik=-7214.281  g=0.955  v=0.116  theta=0.795  sigma2_eps=0.0

In [ ]:
#regressione di PM2.5 in due scenari 
y_previsione_10 = tl.zeros([n_staz,periodo])
y_previsione_nox = tl.zeros([n_staz,periodo])
y_previsione_co = tl.zeros([n_staz,periodo])
y_previsione_benzene = tl.zeros([n_staz,periodo])
y_previsione_o3 = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione_10[:,t] = X_b[:,:,t]@modello_10.beta_ + modello_10.a_filt[t]
    y_previsione_nox[:,t] = X_b[:,:,t]@modello_nox.beta_ + modello_nox.a_filt[t]
    y_previsione_co[:,t] = X_b[:,:,t]@modello_co.beta_ + modello_co.a_filt[t]
    y_previsione_benzene[:,t] = X_b[:,:,t]@modello_benzene.beta_ + modello_benzene.a_filt[t]
    y_previsione_o3[:,t] = X_b[:,:,t]@modello_o3.beta_ + modello_o3.a_filt[t]

X_b_inq = tl.zeros([n_staz, n_regr+1+2+11+5, periodo])
X_b_inq[:,:-5,:] = X_b
X_b_inq[:,-5,:] = y_previsione_10
X_b_inq[:,-4,:] = y_previsione_nox
X_b_inq[:,-3,:] = y_previsione_co
X_b_inq[:,-2,:] = y_previsione_benzene
X_b_inq[:,-1,:] = y_previsione_o3

print('PM2.5 con inquinanti')
modello_inq = HDGM(max_iter=500, tol = 1e-7, previsione= 'PM2o5_media', misure = ['1']+ c_misure+ ['vel10','vel100']+col_mens+['PM10','NOX','CO','Benzene','O3'],metodo='gradiente')
modello_inq.fit(X_b_inq, y_2o5, coord)
modello_inq.summary(X_b_inq,y_2o5, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+['PM10','NOX','CO','Benzene','O3',"g", "v", "theta", "sigma2_eps"])

print('')
print('PM2.5 senza inquinanti')
modello_2o5 = HDGM(max_iter=500, tol = 1e-7, previsione= 'PM2o5_media', misure = ['1']+ c_misure+ ['vel10','vel100']+col_mens,metodo='gradiente')
modello_2o5.fit(X_b, y_2o5, coord)
modello_2o5.summary(X_b,y_2o5, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])



PM2.5 con inquinanti
iter   0  loglik=-75359.767  g=0.607  v=1.163  theta=0.551  sigma2_eps=0.338
iter   1  loglik=-54267.613  g=0.652  v=0.924  theta=0.519  sigma2_eps=0.171
iter   2  loglik=-43318.133  g=0.704  v=0.687  theta=0.467  sigma2_eps=0.103
iter   3  loglik=-35241.131  g=0.765  v=0.490  theta=0.402  sigma2_eps=0.070
iter   4  loglik=-28625.870  g=0.826  v=0.339  theta=0.330  sigma2_eps=0.051
iter   5  loglik=-22983.427  g=0.880  v=0.237  theta=0.268  sigma2_eps=0.039
iter   6  loglik=-18308.683  g=0.920  v=0.175  theta=0.226  sigma2_eps=0.032
iter   7  loglik=-14851.414  g=0.944  v=0.139  theta=0.202  sigma2_eps=0.028
iter   8  loglik=-12615.927  g=0.958  v=0.118  theta=0.192  sigma2_eps=0.025
iter   9  loglik=-11235.618  g=0.966  v=0.105  theta=0.189  sigma2_eps=0.024
iter  10  loglik=-10325.942  g=0.970  v=0.096  theta=0.190  sigma2_eps=0.023
iter  11  loglik=-9651.265  g=0.973  v=0.090  theta=0.193  sigma2_eps=0.023
iter  12  loglik=-9094.099  g=0.975  v=0.085  theta=0.19

In [ ]:
#R^2 sui due modelli di PM2.5, previsioni fino all'istante presente

y_previsione_2o5 = tl.zeros([n_staz,periodo])
y_previsione_2o5_inq = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione_2o5[:,t] = X_b[:,:,t]@modello_2o5.beta_+modello_2o5.a_filt[t]
    y_previsione_2o5_inq[:,t] = X_b_inq[:,:,t]@modello_inq.beta_ + modello_inq.a_filt[t]

valid = ~np.isnan(y_2o5)
R2_2o5 = 1- np.sum((y_2o5[valid]-y_previsione_2o5[valid])**2)/np.sum((y_2o5[valid]-np.mean(y_2o5[valid]))**2)
R2_2o5_inq = 1- np.sum((y_2o5[valid]-y_previsione_2o5_inq[valid])**2)/np.sum((y_2o5[valid]-np.mean(y_2o5[valid]))**2)

print(R2_2o5, 'R^2 del modello senza gli altri inquinanti')

print(R2_2o5_inq, 'R^2 del modello con gli altri inquinanti')

0.8894958166446628 R^2 del modello senza gli altri inquinanti
0.8752177853623124 R^2 del modello con gli altri inquinanti


In [4]:

#confronti tra inquinanti, NOX

c_misure = ['u10_media','v10_media','u100_media','v100_media','tp_media','d2m_media','t2m_media','blh_media','bovini','ovini','suini','pollame','veg_alta','veg_bassa','perc_suolo','rh']

#costruzione della matrice dei coefficienti con le covariate significative (p<0.05) e inserendo regressori categorici per le date

tab['mese'] = tab['data'].dt.month
regr_mens = pd.get_dummies(tab['mese'], prefix='mese',drop_first=True).astype(float)
tab_mens = pd.concat([tab,regr_mens],axis=1)
col_mens = list(regr_mens.columns)


stazioni = list(set(tab['stazione']))
stazioni.sort()

istanti = list(set(tab['data']))
istanti.sort()

n_staz = len(stazioni)
periodo = len(istanti)
c_misure.remove('u100_media')
c_misure.remove('v100_media')
c_misure.remove('u10_media')
c_misure.remove('v10_media')

n_regr = len(c_misure)+1+2+11

X_b = tl.zeros([n_staz, n_regr,periodo])
y_nox = tl.zeros([n_staz, periodo])
y_no2 = tl.zeros([n_staz, periodo])
y_no=  tl.zeros([n_staz, periodo])
y_so2 =  tl.zeros([n_staz, periodo])
y_co = tl.zeros([n_staz, periodo])
y_o3=  tl.zeros([n_staz, periodo])
coord = tl.zeros([n_staz, 2])


for staz, ind_s in zip(stazioni,list(range(0,n_staz))):
    coord[ind_s,:] = [tab[tab['stazione']==staz][c_geo[2]].to_numpy()[0], tab[tab['stazione']==staz][c_geo[3]].to_numpy()[0]]
    y_nox[ind_s, :] = tab[tab['stazione']==staz]['NOX_media'].to_numpy()
    y_no2[ind_s, :] = tab[tab['stazione']==staz]['NO2_media'].to_numpy()
    y_no[ind_s, :] = tab[tab['stazione']==staz]['NO_media'].to_numpy()
    y_so2[ind_s, :] = tab[tab['stazione']==staz]['SO2_media'].to_numpy()
    y_co[ind_s, :] = tab[tab['stazione']==staz]['CO_media'].to_numpy()
    y_o3[ind_s, :] = tab[tab['stazione']==staz]['O3_media'].to_numpy()

    vel10 = (tab[tab['stazione']==staz]['u10_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v10_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)
    vel100 = (tab[tab['stazione']==staz]['u100_media'].apply(lambda x: x**2) +tab[tab['stazione']==staz]['v100_media'].apply(lambda x: x**2)).apply(lambda x: x**0.5)

    X_b[ind_s,0,:] = np.ones(periodo)
    X_b[ind_s,1:-13,:] = tab[tab['stazione']==staz][c_misure].to_numpy().transpose()
    X_b[ind_s,-13,:] = vel10.to_numpy()
    X_b[ind_s,-12,:] = vel100.to_numpy()
    X_b[ind_s,-11:,:] = regr_mens.to_numpy().transpose()[:,:1826]


y_nox =  np.log(y_nox)
y_no2 = np.log(y_no2)
y_no =np.log(y_no)
y_co=  np.log(y_co)
y_so2 = np.log(y_so2)
y_o3= np.log(y_o3)

In [5]:
print('NO')
modello_no = HDGM(max_iter=500, tol = 1e-6, previsione= 'NO_media', misure = ['1']+ c_misure+ ['vel10', 'vel100']+col_mens,metodo='gradiente')
modello_no.fit(X_b, y_no, coord)
modello_no.summary(X_b,y_no, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])


print('')
print('CO')
modello_co = HDGM(max_iter=500, tol = 1e-6, previsione= 'CO_media', misure = ['1']+ c_misure+ ['vel10', 'vel100']+col_mens,metodo='gradiente')
modello_co.fit(X_b, y_co, coord)
modello_co.summary(X_b,y_co, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('SO2')
modello_so2 = HDGM(max_iter=500, tol = 1e-6, previsione= 'SO2_media', misure = ['1']+ c_misure+ ['vel10', 'vel100']+col_mens,metodo='gradiente')
modello_so2.fit(X_b, y_so2, coord)
modello_so2.summary(X_b,y_so2, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('NO2')
modello_no2 = HDGM(max_iter=500, tol = 1e-6, previsione= 'NO2_media', misure = ['1']+ c_misure+ ['vel10', 'vel100']+col_mens,metodo='gradiente')
modello_no2.fit(X_b, y_no2, coord)
modello_no2.summary(X_b,y_no2, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('O3')
modello_o3 = HDGM(max_iter=500, tol = 1e-6, previsione= 'O3_media', misure = ['1']+ c_misure+ ['vel10', 'vel100']+col_mens,metodo='gradiente')
modello_o3.fit(X_b, y_o3, coord)
modello_o3.summary(X_b,y_o3, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])



NO
iter   0  loglik=-71351.004  g=0.560  v=0.829  theta=0.364  sigma2_eps=0.556
iter   1  loglik=-63938.569  g=0.650  v=0.665  theta=0.297  sigma2_eps=0.378
iter   2  loglik=-58377.002  g=0.750  v=0.534  theta=0.241  sigma2_eps=0.270
iter   3  loglik=-52664.016  g=0.830  v=0.437  theta=0.200  sigma2_eps=0.198
iter   4  loglik=-47932.425  g=0.880  v=0.368  theta=0.174  sigma2_eps=0.155
iter   5  loglik=-45197.168  g=0.906  v=0.321  theta=0.158  sigma2_eps=0.131
iter   6  loglik=-43869.995  g=0.920  v=0.290  theta=0.148  sigma2_eps=0.118
iter   7  loglik=-43219.556  g=0.929  v=0.268  theta=0.143  sigma2_eps=0.111
iter   8  loglik=-42871.969  g=0.934  v=0.253  theta=0.139  sigma2_eps=0.108
iter   9  loglik=-42660.514  g=0.938  v=0.242  theta=0.138  sigma2_eps=0.106
iter  10  loglik=-42512.830  g=0.940  v=0.233  theta=0.137  sigma2_eps=0.106
iter  11  loglik=-42397.910  g=0.943  v=0.226  theta=0.137  sigma2_eps=0.106
iter  12  loglik=-42302.323  g=0.944  v=0.220  theta=0.137  sigma2_eps=0.

In [ ]:
#regressione di NOX in due scenari 
y_previsione_no = tl.zeros([n_staz,periodo])
y_previsione_no2 = tl.zeros([n_staz,periodo])
y_previsione_co = tl.zeros([n_staz,periodo])
y_previsione_so2 = tl.zeros([n_staz,periodo])
y_previsione_o3 = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione_no[:,t] = X_b[:,:,t]@modello_no.beta_ + modello_no.a_filt[t]
    y_previsione_no2[:,t] = X_b[:,:,t]@modello_no2.beta_ + modello_no2.a_filt[t]
    y_previsione_co[:,t] = X_b[:,:,t]@modello_co.beta_ + modello_co.a_filt[t]
    y_previsione_so2[:,t] = X_b[:,:,t]@modello_so2.beta_ + modello_so2.a_filt[t]
    y_previsione_o3[:,t] = X_b[:,:,t]@modello_o3.beta_ + modello_o3.a_filt[t]

X_b_inq = tl.zeros([n_staz, n_regr+5, periodo])
X_b_inq[:,:-5,:] = X_b
X_b_inq[:,-5,:] = y_previsione_no
X_b_inq[:,-4,:] = y_previsione_no2
X_b_inq[:,-3,:] = y_previsione_co
X_b_inq[:,-2,:] = y_previsione_so2
X_b_inq[:,-1,:] = y_previsione_o3

print('NOX senza inquinanti')
modello_nox = HDGM(max_iter=500, tol = 1e-7, previsione= 'NOX_media', misure = ['1']+ c_misure+ ['vel10','vel100']+col_mens,metodo='gradiente')
modello_nox.fit(X_b, y_nox, coord)
modello_nox.summary(X_b,y_nox, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+["g", "v", "theta", "sigma2_eps"])

print('')
print('NOX con inquinanti')
modello_inq = HDGM(max_iter=500, tol = 1e-7, previsione= 'NOX_media', misure = ['1']+ c_misure+ ['vel10','vel100']+col_mens+['NO','NO2','CO','SO2','O3'],metodo='gradiente')
modello_inq.fit(X_b_inq, y_nox, coord)
modello_inq.summary(X_b_inq,y_nox, param_names = ['1']+ c_misure+ ['vel10','vel100']+col_mens+['NO','NO2','CO','SO2','O3',"g", "v", "theta", "sigma2_eps"])

NOX senza inquinanti
iter   0  loglik=-141092.981  g=0.683  v=1.118  theta=0.547  sigma2_eps=0.373
iter   1  loglik=-102837.329  g=0.795  v=0.630  theta=0.380  sigma2_eps=0.188
iter   2  loglik=-78759.705  g=0.894  v=0.339  theta=0.256  sigma2_eps=0.109
iter   3  loglik=-59195.354  g=0.945  v=0.222  theta=0.218  sigma2_eps=0.072
iter   4  loglik=-46916.692  g=0.963  v=0.176  theta=0.223  sigma2_eps=0.055
iter   5  loglik=-40551.311  g=0.969  v=0.156  theta=0.245  sigma2_eps=0.046
iter   6  loglik=-37366.378  g=0.971  v=0.146  theta=0.273  sigma2_eps=0.042
iter   7  loglik=-35739.913  g=0.973  v=0.139  theta=0.299  sigma2_eps=0.040
iter   8  loglik=-34818.765  g=0.974  v=0.134  theta=0.323  sigma2_eps=0.040
iter   9  loglik=-34200.777  g=0.975  v=0.130  theta=0.343  sigma2_eps=0.040
iter  10  loglik=-33719.554  g=0.976  v=0.126  theta=0.362  sigma2_eps=0.041
iter  11  loglik=-33312.118  g=0.977  v=0.122  theta=0.378  sigma2_eps=0.042
iter  12  loglik=-32954.480  g=0.978  v=0.118  theta=

In [ ]:
#R^2 sui due modelli di NOX, previsioni fino all'istante presente

y_previsione_nox = tl.zeros([n_staz,periodo])
y_previsione_nox_inq = tl.zeros([n_staz,periodo])
for t in range(periodo):
    y_previsione_nox[:,t] = X_b[:,:,t]@modello_nox.beta_+modello_nox.a_filt[t]
    y_previsione_nox_inq[:,t] = X_b_inq[:,:,t]@modello_inq.beta_ + modello_inq.a_filt[t]

valid = ~np.isnan(y_nox)
R2_nox = 1- np.sum((y_nox[valid]-y_previsione_nox[valid])**2)/np.sum((y_nox[valid]-np.mean(y_nox[valid]))**2)
R2_nox_inq = 1- np.sum((y_nox[valid]-y_previsione_nox_inq[valid])**2)/np.sum((y_nox[valid]-np.mean(y_nox[valid]))**2)

print(R2_nox, 'R^2 del modello senza gli altri inquinanti')

print(R2_nox_inq, 'R^2 del modello con gli altri inquinanti')

0.9187814867963378 R^2 del modello senza gli altri inquinanti
0.9239803732822363 R^2 del modello con gli altri inquinanti
